# Set Cover

All fourteen source stages, eight original subsets covering ten elements, eight decision bits and fifteen encoded slack bits, three QAOA layers, penalty 10, sixty optimizer iterations, exact lower-tail CVaR at 70%, finite optimized and uniform samples, and independent exhaustive constrained comparison.

Reuse the native qfunc for the original subset table. Sum costs over the complete 23-bit distribution using a bounded integer energy histogram. Local Nelder–Mead with an explicit linear pi schedule replaces the unpinned hosted optimizer. Use 1000 actual CPU draws per comparison. Select the least penalized observed joint word without repairing it. An independent exhaustive search over 256 decision words replaces Couenne after sampling. No historical trajectory equivalence or optimality guarantee.

## Treat modeled slack as part of the search space

The preview exposes three cost/mixer layers over eight selection bits and
fifteen slack bits. The imported cost helper retains the ten element-coverage
constraints, including nonbinary slack weights. FlagQuantum compiles those
phases and named outputs. The user supplies the subsets, encoding and penalty;
the compiler does not discover a covering algorithm.

Try a different six-angle preview binding and inspect coverage separately
from the full joint penalized cost. CVaR0.7 means the lower-cost probability
tail, not a confidence level. Sixty classical optimizer iterations do not
certify a minimum cover. Propagate algorithm edits to the shared main.py used
by run_workflow. The 23-bit complete replay retains all original slack states.


## Define the coverage model

In [ ]:
from flagquantum import synth
from flagquantum.synth import (
    RX,
    Output,
    QArray,
    QBit,
    allocate,
    hadamard_transform,
    qfunc,
)
from examples.synthesis.applications.optimization.set_cover.main import cost


@qfunc
def main(
    params: list[float],
    x: Output[QArray[QBit]],
    s0: Output[QArray[QBit]],
    s1: Output[QArray[QBit]],
    s2: Output[QArray[QBit]],
    s3: Output[QArray[QBit]],
    s4: Output[QArray[QBit]],
    s5: Output[QArray[QBit]],
    s6: Output[QArray[QBit]],
    s7: Output[QArray[QBit]],
    s8: Output[QArray[QBit]],
    s9: Output[QArray[QBit]],
) -> None:
    allocate(8, x)
    allocate(1, s0)
    allocate(2, s1)
    allocate(2, s2)
    allocate(2, s3)
    allocate(1, s4)
    allocate(1, s5)
    allocate(2, s6)
    allocate(2, s7)
    allocate(1, s8)
    allocate(1, s9)
    hadamard_transform(x)
    hadamard_transform(s0)
    hadamard_transform(s1)
    hadamard_transform(s2)
    hadamard_transform(s3)
    hadamard_transform(s4)
    hadamard_transform(s5)
    hadamard_transform(s6)
    hadamard_transform(s7)
    hadamard_transform(s8)
    hadamard_transform(s9)
    for layer in range(3):
        cost(params[layer], x, s0, s1, s2, s3, s4, s5, s6, s7, s8, s9)
        for j in range(8):
            RX(params[3 + layer], x[j])
        for j in range(1):
            RX(params[3 + layer], s0[j])
        for j in range(2):
            RX(params[3 + layer], s1[j])
        for j in range(2):
            RX(params[3 + layer], s2[j])
        for j in range(2):
            RX(params[3 + layer], s3[j])
        for j in range(1):
            RX(params[3 + layer], s4[j])
        for j in range(1):
            RX(params[3 + layer], s5[j])
        for j in range(2):
            RX(params[3 + layer], s6[j])
        for j in range(2):
            RX(params[3 + layer], s7[j])
        for j in range(1):
            RX(params[3 + layer], s8[j])
        for j in range(1):
            RX(params[3 + layer], s9[j])


from pathlib import Path

from IPython.display import Image, display

from examples.synthesis.applications.optimization.set_cover.overview.main import (
    selected_sets,
)
from examples.synthesis.applications.optimization.set_cover.overview.model import (
    source_model,
)
from examples.synthesis.applications.optimization.set_cover.overview.plots import (
    draw_histogram,
    draw_trace,
)
from examples.synthesis.applications.optimization.set_cover.overview.workflow import (
    INITIAL,
    run_workflow,
)

model = source_model()


## Inspect the eight original subsets

In [ ]:
print(model.sets)


## Inspect each coverage constraint

In [ ]:
for element, group in enumerate(model.coverers, start=1):
    print(f"Element {element}: choose at least one of {group}")


## Set the original optimization size

In [ ]:
print("Three layers, penalty 10, sixty iterations, 70% lower-tail CVaR")


## Synthesize with the starting parameters

In [ ]:
print(
    synth.synthesize(
        main,
        classical_inputs={"params": INITIAL},
        max_qubits=23,
        max_workspace=0,
        max_operations=4096,
        memory_limit_bytes=512 * 1024**2,
    ).report["resources"]
)


## Run all iterations

In [ ]:
result = run_workflow()
print("Actual iterations:", result.optimizer.iterations)
print("Actual objective evaluations:", len(result.trace))
print("Converged:", result.optimizer.converged)


## Plot the actual cost trace

In [ ]:
artifact_dir = Path("artifacts/development/set-cover")
display(Image(filename=str(draw_trace(result, artifact_dir / "cost-trace.png"))))


## Inspect the five lowest observed penalized costs

In [ ]:
for row in result.optimized[:5]:
    print(row)


## Check the uniform sample size

In [ ]:
print("Uniform draws:", sum(row.count for row in result.uniform))


## Compare the sampled cost distributions

In [ ]:
display(
    Image(filename=str(draw_histogram(result, artifact_dir / "cost-histogram.png")))
)


## Select the least penalized observed state

In [ ]:
best = result.optimized[0]
print("Best actually observed:", best)


## Read the selected subsets

In [ ]:
print("Observed subsets:", selected_sets(model, best.decisions))


## Inspect the independent constrained minimum

In [ ]:
print("Exact minimum words:", result.classical)


## Read one optimal subset selection

In [ ]:
print("One exact solution:", selected_sets(model, result.classical[0]))
